import library



In [16]:
import pandas as pd
import numpy as np

In [17]:
# data loading
train_df = pd.read_csv("../data/raw/train.csv")
store_df = pd.read_csv("../data/raw/store.csv")

print(train_df.shape)
print(store_df.shape)

(1017209, 9)
(1115, 10)


C:\Users\kztam\AppData\Local\Temp\ipykernel_29480\2788861920.py:2: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("../data/raw/train.csv")


In [18]:
train_df["Date"] = pd.to_datetime(train_df["Date"])
print(train_df["Date"].dtype)

datetime64[us]


In [19]:
#sort data
train_df = train_df.sort_values(["Store", "Date"]).reset_index(drop=True)
display(train_df.head())

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,2,2013-01-01,0,0,0,0,a,1
1,1,3,2013-01-02,5530,668,1,0,0,1
2,1,4,2013-01-03,4327,578,1,0,0,1
3,1,5,2013-01-04,4486,619,1,0,0,1
4,1,6,2013-01-05,4997,635,1,0,0,1


In [20]:
#marge store inforamation
forecast_df = train_df.merge(store_df, on="Store", how="left")
print(forecast_df.shape)
forecast_df.head()

(1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,2,2013-01-01,0,0,0,0,a,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,1,3,2013-01-02,5530,668,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
2,1,4,2013-01-03,4327,578,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
3,1,5,2013-01-04,4486,619,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
4,1,6,2013-01-05,4997,635,1,0,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN


In [21]:
# create time features

forecast_df["Year"] = forecast_df["Date"].dt.year
forecast_df["Month"] = forecast_df["Date"].dt.month
forecast_df["Day"] = forecast_df["Date"].dt.day
forecast_df["DayOfWeek"] = forecast_df["Date"].dt.dayofweek
forecast_df["WeekOfYear"] = forecast_df["Date"].dt.isocalendar().week.astype(int)

Create weekend feature

In [22]:
forecast_df["IsWeekend"] = (
    forecast_df["DayOfWeek"] >= 5
).astype(int)


In [23]:
forecast_df["Sales_Lag_1"] = (
    forecast_df.groupby("Store")["Sales"].shift(1)
)

forecast_df["Sales_Lag_7"] = (
    forecast_df.groupby("Store")["Sales"].shift(7)
)

forecast_df["Sales_Lag_14"] = (
    forecast_df.groupby("Store")["Sales"].shift(14)
)

Create rolling averages

In [24]:
forecast_df["Sales_Rolling_7"] = (
    forecast_df.groupby("Store")["Sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

forecast_df["Sales_Rolling_14"] = (
    forecast_df.groupby("Store")["Sales"]
    .transform(lambda x: x.shift(1).rolling(14).mean())
)

Check data set

In [26]:
print(forecast_df.shape)
print(forecast_df.columns.tolist())

forecast_df.head(10)

(1017209, 28)
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Rolling_7', 'Sales_Rolling_14']


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Year,Month,Day,WeekOfYear,IsWeekend,Sales_Lag_1,Sales_Lag_7,Sales_Lag_14,Sales_Rolling_7,Sales_Rolling_14
0,1,1,2013-01-01,0,0,0,0,a,1,c,...,2013,1,1,1,0,NaN,NaN,NaN,NaN,NaN
1,1,2,2013-01-02,5530,668,1,0,0,1,c,...,2013,1,2,1,0,0.0,NaN,NaN,NaN,NaN
2,1,3,2013-01-03,4327,578,1,0,0,1,c,...,2013,1,3,1,0,5530.0,NaN,NaN,NaN,NaN
3,1,4,2013-01-04,4486,619,1,0,0,1,c,...,2013,1,4,1,0,4327.0,NaN,NaN,NaN,NaN
4,1,5,2013-01-05,4997,635,1,0,0,1,c,...,2013,1,5,1,1,4486.0,NaN,NaN,NaN,NaN
5,1,6,2013-01-06,0,0,0,0,0,1,c,...,2013,1,6,1,1,4997.0,NaN,NaN,NaN,NaN
6,1,0,2013-01-07,7176,785,1,1,0,1,c,...,2013,1,7,2,0,0.0,NaN,NaN,NaN,NaN
7,1,1,2013-01-08,5580,654,1,1,0,1,c,...,2013,1,8,2,0,7176.0,0.0,NaN,3788.000000,NaN
8,1,2,2013-01-09,5471,626,1,1,0,1,c,...,2013,1,9,2,0,5580.0,5530.0,NaN,4585.142857,NaN
9,1,3,2013-01-10,4892,615,1,1,0,1,c,...,2013,1,10,2,0,5471.0,4327.0,NaN,4576.714286,NaN


Check missing values


In [27]:
forecast_df[
    [
        "Sales_Lag_1",
        "Sales_Lag_7",
        "Sales_Lag_14",
        "Sales_Rolling_7",
        "Sales_Rolling_14"
    ]
].isnull().sum()

Sales_Lag_1          1115
Sales_Lag_7          7805
Sales_Lag_14        15610
Sales_Rolling_7      7805
Sales_Rolling_14    15610
dtype: int64

Remove Row with insufficient history

In [30]:
forecast_df = forecast_df.dropna(
    subset=[
        "Sales_Lag_1",
        "Sales_Lag_7",
        "Sales_Lag_14",
        "Sales_Rolling_7",
        "Sales_Rolling_14"
    ]
).reset_index(drop=True)


print(forecast_df.shape)
display(forecast_df.head(10))

(1001599, 28)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Year,Month,Day,WeekOfYear,IsWeekend,Sales_Lag_1,Sales_Lag_7,Sales_Lag_14,Sales_Rolling_7,Sales_Rolling_14
0,1,1,2013-01-15,3900,512,1,0,0,0,c,...,2013,1,15,3,0,4717.0,5580.0,0.0,4356.142857,4072.071429
1,1,2,2013-01-16,4008,530,1,0,0,0,c,...,2013,1,16,3,0,3900.0,5471.0,5530.0,4116.142857,4350.642857
2,1,3,2013-01-17,4044,503,1,0,0,0,c,...,2013,1,17,3,0,4008.0,4892.0,4327.0,3907.142857,4241.928571
3,1,4,2013-01-18,4127,568,1,0,0,0,c,...,2013,1,18,3,0,4044.0,4881.0,4486.0,3786.000000,4221.714286
4,1,5,2013-01-19,5182,688,1,0,0,0,c,...,2013,1,19,3,1,4127.0,4952.0,4997.0,3678.285714,4196.071429
5,1,6,2013-01-20,0,0,0,0,0,0,c,...,2013,1,20,3,1,5182.0,0.0,0.0,3711.142857,4209.285714
6,1,0,2013-01-21,5394,607,1,1,0,0,c,...,2013,1,21,4,0,0.0,4717.0,7176.0,3711.142857,4209.285714
7,1,1,2013-01-22,5720,667,1,1,0,0,c,...,2013,1,22,4,0,5394.0,3900.0,5580.0,3807.857143,4082.000000
8,1,2,2013-01-23,5578,627,1,1,0,0,c,...,2013,1,23,4,0,5720.0,4008.0,5471.0,4067.857143,4092.000000
9,1,3,2013-01-24,5195,645,1,1,0,0,c,...,2013,1,24,4,0,5578.0,4044.0,4892.0,4292.142857,4099.642857


Save forcasting data set

In [31]:
forecast_df.to_csv(
    "../data/processed/forecasting_dataset.csv",
    index=False
)